# Bottom Detection for EK80 Echograms

Applies bottom detection to EK80 `.raw` files using algorithms ported from
[CRIMAC-classifiers-bottom](https://github.com/CRIMAC-WP4-Machine-learning/CRIMAC-classifiers-bottom) (MIT license).

Two detection algorithms are used: `simple` (argmax + back-step to leading edge) and
`edge` (Gaussian-derivative convolution + peak ranking, works well on flat bottom like the Oslofjord data).
A third gradient-based detector and an ensemble vote combine them.

Output includes bottom depth overlays, binary segmentation masks, and per-file CSV exports.

In [ ]:
import sys
sys.path.insert(0, '../..')

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

import importlib
from raw_sensor_integration import ping_parser, metadata, raw_parser, bottom_detection
importlib.reload(bottom_detection)

## 1. Load a raw file

In [ ]:
RAW_DIR = Path.home() / 'Downloads' / 'drive-download-20251116T003920Z-1-001'
raw_files = sorted(RAW_DIR.glob('D20240131*.raw'))
print(f"Found {len(raw_files)} raw files:")
for i, f in enumerate(raw_files):
    print(f"  [{i}] {f.name}")

In [ ]:
FILE_IDX = 0
MAX_PINGS = 2000   # Increase for batch processing; 2000 ≈ ~17 min of data at 2 Hz

sample_file = raw_files[FILE_IDX]
print(f"Loading: {sample_file.name}")

pings = ping_parser.extract_pings(str(sample_file), max_pings=MAX_PINGS)
print(f"Extracted {len(pings)} total pings")

# Identify channels
channel_ids = list(dict.fromkeys(p.channel_id for p in pings))  # preserve order
print("\nChannels:")
for i, ch in enumerate(channel_ids):
    ch_pings = [p for p in pings if p.channel_id == ch]
    print(f"  [{i}] {ch}")
    print(f"      {len(ch_pings)} pings, {ch_pings[0].sample_count} samples/ping")

## 2. Build range array and extract pulse duration

The long-range channel (2768 samples, ~200 m) is the primary target for bottom detection.
The short-range channel (~240 samples, ~23 m) only reaches the bottom if it is shallower than its max range.

In [ ]:
ch_settings = metadata.get_channel_settings(str(sample_file))

print("Channel settings:")
for s in ch_settings:
    print(f"  {s['channel_id'][:50]}")
    print(f"    sample_count={s.get('sample_count','?')}  "
          f"sample_interval={s['sample_interval']:.2e} s  "
          f"max_range={s.get('max_range','?'):.1f} m")

# Pick the long-range channel (most samples)
long_range_settings = max(
    [s for s in ch_settings if 'sample_count' in s],
    key=lambda s: s['sample_count']
)
SELECTED_CH = long_range_settings['channel_id']
print(f"\nSelected channel: {SELECTED_CH}")

range_array = bottom_detection.build_range_array(
    long_range_settings['sample_count'],
    long_range_settings['sample_interval'],
)
sample_dist = float(range_array[1] - range_array[0])
print(f"Range: {range_array[0]:.3f} – {range_array[-1]:.1f} m  "
      f"(step {sample_dist*100:.2f} cm/sample)")

pulse_duration = metadata.get_pulse_duration(str(sample_file), SELECTED_CH)
print(f"Pulse duration: {pulse_duration*1000:.4f} ms")

In [ ]:
# Build power image for selected channel
power_db, ping_times = ping_parser.get_ping_matrix(pings, SELECTED_CH)
# get_ping_matrix returns complex samples; convert to dB
power_linear = np.abs(power_db) ** 2
power_linear = np.maximum(power_linear, 1e-20)
power_db_array = 10 * np.log10(power_linear).astype(np.float32)
power_db_array = np.clip(power_db_array, -100, 0)

# Alternatively, use pings_to_power_image (returns only the 2D array, timestamps via get_ping_matrix)
n_samples, n_pings = power_db_array.shape
print(f"Echogram shape: {n_samples} samples × {n_pings} pings")

# Confirm range array matches
if len(range_array) != n_samples:
    print(f"WARNING: range_array length ({len(range_array)}) != n_samples ({n_samples})")
    range_array = range_array[:n_samples]  # trim if needed

In [ ]:
# Diagnostic: inspect power distribution to understand threshold requirements
print(f"Power dB range: {power_db_array.min():.1f} to {power_db_array.max():.1f}")
print(f"Percentiles: 25%={np.percentile(power_db_array, 25):.1f}, "
      f"50%={np.percentile(power_db_array, 50):.1f}, "
      f"75%={np.percentile(power_db_array, 75):.1f}, "
      f"95%={np.percentile(power_db_array, 95):.1f}, "
      f"99%={np.percentile(power_db_array, 99):.1f}")

# Per-ping max — the strongest return per ping (typically the bottom echo)
ping_maxes = power_db_array.max(axis=0)
print(f"\nPer-ping max dB: min={ping_maxes.min():.1f}, "
      f"median={np.median(ping_maxes):.1f}, max={ping_maxes.max():.1f}")
print(f"Pings with max > -31 dB (old threshold): {(ping_maxes > -31).sum()}/{n_pings} "
      f"({(ping_maxes > -31).mean():.1%})")
print("\nNote: CRIMAC threshold of -31 dB assumes calibrated Sv.")
print("With uncalibrated raw power, adaptive threshold ('auto') is used instead.")

## 3. Parse MRU0 heave

Heave from the motion reference unit corrects for vessel motion when converting range to depth.
For the Oslofjord data in January the heave is small, but included for completeness.

In [ ]:
raw_data = raw_parser.read_raw_file(str(sample_file))
datagram_types = raw_parser.find_datagram_types(raw_data)

if 'MRU0' in datagram_types:
    mru_data = bottom_detection.parse_mru0_datagrams(raw_data, datagram_types['MRU0'])
    if mru_data:
        mru_times, heave_vals = zip(*mru_data)
        heave_per_ping = bottom_detection.interpolate_heave_to_pings(
            list(mru_times), np.array(heave_vals), ping_times
        )
        print(f"MRU0 data: {len(mru_data)} records")
        print(f"Heave range: {heave_per_ping.min():.3f} – {heave_per_ping.max():.3f} m  "
              f"(std={heave_per_ping.std():.3f} m)")
    else:
        print("MRU0 offsets found but parsing returned no valid records — using zero heave")
        heave_per_ping = np.zeros(n_pings)
else:
    print("No MRU0 datagrams found — using zero heave")
    heave_per_ping = np.zeros(n_pings)

# Trim / pad heave to match n_pings
if len(heave_per_ping) != n_pings:
    heave_per_ping = np.resize(heave_per_ping, n_pings)

## 4. Simple bottom detection

In [ ]:
bottom_simple = bottom_detection.detect_bottom_simple(
    power_db=power_db_array,
    range_array=range_array,
    heave=heave_per_ping,
    pulse_duration=pulse_duration,
    transducer_draft=0.0,
    minimum_range=10.0,
    offset=0.5,
)

valid_frac = np.sum(~np.isnan(bottom_simple)) / n_pings
print("Simple detection (adaptive threshold):")
print(f"  Valid pings: {valid_frac:.1%} ({np.sum(~np.isnan(bottom_simple))}/{n_pings})")
print(f"  Depth range: {np.nanmin(bottom_simple):.1f} – {np.nanmax(bottom_simple):.1f} m")
print(f"  Median depth: {np.nanmedian(bottom_simple):.1f} m")

## 5. Edge bottom detection (more accurate for flat bottom)

In [ ]:
# Determine frequency for beam-width correction
# ES38-18 = 38 kHz transducer
freq_hz = long_range_settings.get('frequency', 38000)

bottom_edge = bottom_detection.detect_bottom_edge(
    power_db=power_db_array,
    range_array=range_array,
    heave=heave_per_ping,
    pulse_duration=pulse_duration,
    frequency_hz=freq_hz,
    transducer_draft=0.0,
    minimum_range=10.0,
    offset=0.5,
)

valid_frac_e = np.sum(~np.isnan(bottom_edge)) / n_pings
print("Edge detection (adaptive threshold):")
print(f"  Valid pings: {valid_frac_e:.1%} ({np.sum(~np.isnan(bottom_edge))}/{n_pings})")
print(f"  Depth range: {np.nanmin(bottom_edge):.1f} – {np.nanmax(bottom_edge):.1f} m")
print(f"  Median depth: {np.nanmedian(bottom_edge):.1f} m")

# Agreement between algorithms
both_valid = ~np.isnan(bottom_simple) & ~np.isnan(bottom_edge)
diff = bottom_simple[both_valid] - bottom_edge[both_valid]
print("\nAgreement (simple − edge):")
print(f"  Mean diff: {diff.mean():.2f} m  Std: {diff.std():.2f} m  Max|diff|: {np.abs(diff).max():.1f} m")

## 5b. Gradient bottom detection

Threshold-free approach — uses the steepest power gradient in the depth profile.

In [ ]:
bottom_gradient = bottom_detection.detect_bottom_gradient(
    power_db=power_db_array,
    range_array=range_array,
    minimum_range=10.0,
    gradient_window=5,
    offset=0.5,
)

valid_frac_g = np.sum(~np.isnan(bottom_gradient)) / n_pings
print("Gradient detection:")
print(f"  Valid pings: {valid_frac_g:.1%} ({np.sum(~np.isnan(bottom_gradient))}/{n_pings})")
print(f"  Depth range: {np.nanmin(bottom_gradient):.1f} – {np.nanmax(bottom_gradient):.1f} m")
print(f"  Median depth: {np.nanmedian(bottom_gradient):.1f} m")

## 5c. Ensemble: median vote of all 3 detectors

In [ ]:
bottom_ensemble = bottom_detection.ensemble_bottom(
    [bottom_simple, bottom_edge, bottom_gradient],
    max_deviation=5.0,
    primary_count=2,  # simple + edge are CRIMAC-ported; gradient is supplemental
)

valid_frac_ens = np.sum(~np.isnan(bottom_ensemble)) / n_pings
print("Ensemble (primary_count=2, max_deviation=5m):")
print(f"  Valid pings: {valid_frac_ens:.1%} ({np.sum(~np.isnan(bottom_ensemble))}/{n_pings})")
print(f"  Depth range: {np.nanmin(bottom_ensemble):.1f} – {np.nanmax(bottom_ensemble):.1f} m")
print(f"  Median depth: {np.nanmedian(bottom_ensemble):.1f} m")

# Per-algorithm comparison
print("\nComparison:")
print(f"  {'Detector':<12} {'Valid':>8} {'Median':>8} {'Min':>8} {'Max':>8}")
print(f"  {'-'*44}")
for name, arr in [('Simple', bottom_simple), ('Edge', bottom_edge),
                   ('Gradient', bottom_gradient), ('Ensemble', bottom_ensemble)]:
    v = np.sum(~np.isnan(arr)) / n_pings
    print(f"  {name:<12} {v:>7.1%} {np.nanmedian(arr):>7.1f}m {np.nanmin(arr):>7.1f}m {np.nanmax(arr):>7.1f}m")

## 5d. Temporal post-processing

Sliding-window median filter to reject single-ping outliers (e.g. bubble layers at survey start).
The 31-ping window makes the reference robust even when the first ~15 pings are wrong.

In [ ]:
bottom_final, outlier_mask = bottom_detection.post_process_bottom(
    bottom_ensemble,
    max_deviation_m=10.0,   # reject detections >10 m from local running median
    window=31,               # 31-ping wide reference window
    fill_gaps=True,          # interpolate over removed samples
)

n_removed = outlier_mask.sum()
print(f"Post-processing removed {n_removed} outlier pings ({n_removed/n_pings:.1%})")
if n_removed > 0:
    removed_indices = np.where(outlier_mask)[0]
    print(f"  Removed at pings: {removed_indices[:10].tolist()}"
          f"{'...' if n_removed > 10 else ''}")
print(f"Final bottom range: {np.nanmin(bottom_final):.1f} – {np.nanmax(bottom_final):.1f} m")
print(f"Final median depth: {np.nanmedian(bottom_final):.1f} m")

# Side-by-side: before and after post-processing
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
vmin, vmax = np.percentile(power_db_array, [2, 98])

for ax, (depths, title) in zip(axes, [
    (bottom_ensemble, 'Ensemble (before post-processing)'),
    (bottom_final,    'Ensemble + temporal post-processing'),
]):
    ax.imshow(power_db_array, aspect='auto', cmap='viridis',
              vmin=vmin, vmax=vmax, origin='upper')
    bottom_detection.overlay_bottom_on_echogram(
        ax, depths, range_array, color='red', linewidth=1.5, label=title
    )
    ax.set_ylabel('Depth sample')
    ax.set_title(f'{title}  (valid: {np.mean(~np.isnan(depths)):.1%})')
    ax.legend(loc='upper right')

axes[-1].set_xlabel('Ping number')
plt.suptitle(f'{sample_file.name}', fontsize=10)
plt.tight_layout()
plt.show()

## 6. Visualise: echogram with bottom line overlay

In [ ]:
vmin, vmax = np.percentile(power_db_array, [2, 98])

fig, axes = plt.subplots(4, 1, figsize=(14, 18), sharex=True)

for ax, (depths, label) in zip(axes, [
    (bottom_simple,   'Simple'),
    (bottom_edge,     'Edge'),
    (bottom_gradient, 'Gradient'),
    (bottom_ensemble, 'Ensemble'),
]):
    ax.imshow(power_db_array, aspect='auto', cmap='viridis',
              vmin=vmin, vmax=vmax, origin='upper')
    bottom_detection.overlay_bottom_on_echogram(
        ax, depths, range_array, color='red', linewidth=1.5, label=label
    )
    ax.set_ylabel('Depth sample')
    ax.set_title(f'{label} bottom detection  (valid: {np.mean(~np.isnan(depths)):.1%})')
    ax.legend(loc='upper right')

axes[-1].set_xlabel('Ping number')
plt.suptitle(f'{sample_file.name}  –  {SELECTED_CH[:40]}', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side comparison: depth profiles over time
fig, ax = plt.subplots(figsize=(14, 4))

ping_idx = np.arange(n_pings)
ax.plot(ping_idx, bottom_simple, '-', linewidth=0.5, alpha=0.4, color='blue', label='Simple')
ax.plot(ping_idx, bottom_edge, '-', linewidth=0.5, alpha=0.4, color='orange', label='Edge')
ax.plot(ping_idx, bottom_gradient, '-', linewidth=0.5, alpha=0.4, color='green', label='Gradient')
ax.plot(ping_idx, bottom_ensemble, '-', linewidth=1.5, alpha=0.9, color='red', label='Ensemble')
ax.set_xlabel('Ping number')
ax.set_ylabel('Bottom range from transducer (m)')
ax.set_title('Bottom depth profiles: individual detectors (faint) vs ensemble (red)')
ax.invert_yaxis()
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Binary segmentation masks

`'line'` mode marks only the bottom boundary pixels (used for visualization).
`'below'` mode fills everything at or below the detected bottom — this is the U-Net training target.

In [ ]:
# Use ensemble detection results for the final mask
bottom_indices = bottom_detection.bottom_to_sample_index(bottom_ensemble, range_array)

mask_line = bottom_detection.create_bottom_mask(
    n_samples, n_pings, bottom_indices,
    mode='line', line_thickness=3,  # 3-px thick for better U-Net gradient signal
)
mask_below = bottom_detection.create_bottom_mask(
    n_samples, n_pings, bottom_indices,
    mode='below',
)

# Coverage stats
labeled_pings = np.sum(bottom_indices >= 0)
print(f"Mask coverage: {labeled_pings}/{n_pings} pings labeled ({labeled_pings/n_pings:.1%})")
print(f"Mask shapes: line={mask_line.shape}, below={mask_below.shape}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Echogram
axes[0].imshow(power_db_array, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax)
bottom_detection.overlay_bottom_on_echogram(
    axes[0], bottom_ensemble, range_array, color='red', linewidth=1.0, label='Ensemble'
)
axes[0].set_title('Echogram + ensemble detection')
axes[0].set_xlabel('Ping')
axes[0].set_ylabel('Depth sample')
axes[0].legend(loc='upper right', fontsize=8)

# Line mask
axes[1].imshow(mask_line, aspect='auto', cmap='gray_r', vmin=0, vmax=1)
axes[1].set_title('Line mask (3 px thick)')
axes[1].set_xlabel('Ping')
axes[1].set_ylabel('Depth sample')

# Below mask
axes[2].imshow(mask_below, aspect='auto', cmap='gray_r', vmin=0, vmax=1)
axes[2].set_title('Below-bottom mask (U-Net target)')
axes[2].set_xlabel('Ping')
axes[2].set_ylabel('Depth sample')

plt.suptitle(f'{sample_file.name}', fontsize=10)
plt.tight_layout()
plt.show()

## 8. Short-range channel

The second channel (~23 m range) may or may not show bottom depending on
local depth. We check the range and only run detection if depth is reachable.

In [ ]:
short_range_settings = min(
    [s for s in ch_settings if 'sample_count' in s],
    key=lambda s: s['sample_count']
)
SHORT_CH = short_range_settings['channel_id']
short_max_range = short_range_settings.get('max_range', 0)

print(f"Short-range channel: {SHORT_CH[:50]}")
print(f"Max range: {short_max_range:.1f} m")
print(f"Median bottom depth from long-range: {np.nanmedian(bottom_edge):.1f} m")

if short_max_range < np.nanmedian(bottom_edge) * 0.7:
    print("\nShort-range channel does not reach bottom depth — skipping detection.")
    SHORT_RANGE_BOTTOM_DETECTABLE = False
else:
    SHORT_RANGE_BOTTOM_DETECTABLE = True
    print("\nBottom may be detectable on short-range channel — running detection.")

In [ ]:
if SHORT_RANGE_BOTTOM_DETECTABLE:
    short_power_db = ping_parser.pings_to_power_image(pings, SHORT_CH)
    short_power_db = np.clip(short_power_db, -100, 0)

    short_range_array = bottom_detection.build_range_array(
        short_range_settings['sample_count'],
        short_range_settings['sample_interval'],
    )
    short_pulse = metadata.get_pulse_duration(str(sample_file), SHORT_CH)

    # Adjust heave array length
    short_n_pings = short_power_db.shape[1]
    short_heave = heave_per_ping[:short_n_pings] if len(heave_per_ping) >= short_n_pings \
                  else np.zeros(short_n_pings)

    bottom_short = bottom_detection.detect_bottom_edge(
        power_db=short_power_db,
        range_array=short_range_array,
        heave=short_heave,
        pulse_duration=short_pulse,
        frequency_hz=freq_hz,
        minimum_range=2.0,   # shorter minimum for near-range channel
        offset=0.5,
    )

    vmin_s, vmax_s = np.percentile(short_power_db, [2, 98])
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.imshow(short_power_db, aspect='auto', cmap='viridis', vmin=vmin_s, vmax=vmax_s)
    bottom_detection.overlay_bottom_on_echogram(
        ax, bottom_short, short_range_array, color='red', label='Edge'
    )
    valid_s = np.sum(~np.isnan(bottom_short)) / short_n_pings
    ax.set_title(f'Short-range channel bottom detection (valid: {valid_s:.1%})')
    ax.set_xlabel('Ping')
    ax.set_ylabel('Depth sample')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Skipping short-range channel (bottom out of range).")

## 9. Batch processing: all 5 files

In [ ]:
OUTPUT_DIR = Path('.')
results_summary = []

for raw_file in raw_files:
    print(f"\n{'='*60}")
    print(f"Processing: {raw_file.name}")

    try:
        # Load pings
        pings_f = ping_parser.extract_pings(str(raw_file))
        if not pings_f:
            print("  No pings extracted — skipping")
            continue

        # Channel settings
        ch_settings_f = metadata.get_channel_settings(str(raw_file))
        long_ch_f = max(
            [s for s in ch_settings_f if 'sample_count' in s],
            key=lambda s: s['sample_count'],
        )
        selected_ch_f = long_ch_f['channel_id']

        # Power image + timestamps
        pdb_f, ptimes_f = ping_parser.get_ping_matrix(pings_f, selected_ch_f)
        pdb_linear = np.maximum(np.abs(pdb_f) ** 2, 1e-20)
        pdb_f = np.clip(10 * np.log10(pdb_linear), -100, 0).astype(np.float32)
        n_s_f, n_p_f = pdb_f.shape

        range_f = bottom_detection.build_range_array(
            long_ch_f['sample_count'], long_ch_f['sample_interval']
        )
        if len(range_f) != n_s_f:
            range_f = range_f[:n_s_f]

        pulse_f = metadata.get_pulse_duration(str(raw_file), selected_ch_f)

        # Heave
        raw_data_f = raw_parser.read_raw_file(str(raw_file))
        dtypes_f = raw_parser.find_datagram_types(raw_data_f)
        if 'MRU0' in dtypes_f:
            mru_f = bottom_detection.parse_mru0_datagrams(raw_data_f, dtypes_f['MRU0'])
            if mru_f:
                mt_f, hv_f = zip(*mru_f)
                heave_f = bottom_detection.interpolate_heave_to_pings(
                    list(mt_f), np.array(hv_f), ptimes_f
                )
            else:
                heave_f = np.zeros(n_p_f)
        else:
            heave_f = np.zeros(n_p_f)

        # Run all 3 detectors
        b_simple = bottom_detection.detect_bottom_simple(
            power_db=pdb_f, range_array=range_f, heave=heave_f,
            pulse_duration=pulse_f, minimum_range=10.0, offset=0.5,
        )
        b_edge = bottom_detection.detect_bottom_edge(
            power_db=pdb_f, range_array=range_f, heave=heave_f,
            pulse_duration=pulse_f, frequency_hz=long_ch_f.get('frequency', 38000),
            minimum_range=10.0, offset=0.5,
        )
        b_gradient = bottom_detection.detect_bottom_gradient(
            power_db=pdb_f, range_array=range_f, minimum_range=10.0, offset=0.5,
        )

        # Ensemble: simple + edge are primary (CRIMAC-ported); gradient is supplemental
        bottom_raw = bottom_detection.ensemble_bottom(
            [b_simple, b_edge, b_gradient], max_deviation=5.0, primary_count=2,
        )

        # Temporal post-processing: remove pings that disagree with local context
        bottom_f, _ = bottom_detection.post_process_bottom(
            bottom_raw, max_deviation_m=10.0, window=31, fill_gaps=True,
        )

        valid_pct = np.sum(~np.isnan(bottom_f)) / n_p_f
        med_depth = np.nanmedian(bottom_f)

        # Per-detector rates for comparison
        v_s = np.sum(~np.isnan(b_simple)) / n_p_f
        v_e = np.sum(~np.isnan(b_edge)) / n_p_f
        v_g = np.sum(~np.isnan(b_gradient)) / n_p_f
        print(f"  Pings: {n_p_f}  "
              f"Simple: {v_s:.0%}  Edge: {v_e:.0%}  "
              f"Gradient: {v_g:.0%}  Ensemble+PP: {valid_pct:.0%}")
        print(f"  Median depth: {med_depth:.1f} m")

        # Save CSV
        csv_path = OUTPUT_DIR / f"{raw_file.stem}_bottom.csv"
        bottom_detection.save_bottom_detections(
            bottom_f, ptimes_f, selected_ch_f, str(csv_path)
        )
        print(f"  Saved: {csv_path.name}")

        results_summary.append({
            'file': raw_file.name,
            'n_pings': n_p_f,
            'valid_simple': v_s,
            'valid_edge': v_e,
            'valid_gradient': v_g,
            'valid_ensemble': valid_pct,
            'median_depth_m': med_depth,
            'min_depth_m': float(np.nanmin(bottom_f)),
            'max_depth_m': float(np.nanmax(bottom_f)),
        })

    except Exception as e:
        print(f"  ERROR: {e}")
        import traceback
        traceback.print_exc()

print("\nBatch complete.")

## 10. Summary statistics

In [ ]:
import pandas as pd

df = pd.DataFrame(results_summary)
print(df.to_string(index=False, float_format=lambda x: f'{x:.2f}'))

print("\nOverall:")
print(f"  Total pings processed: {df['n_pings'].sum()}")
print(f"  Mean valid rate (ensemble+PP): {df['valid_ensemble'].mean():.1%}")
print(f"  Depth range across all files: {df['min_depth_m'].min():.1f} – {df['max_depth_m'].max():.1f} m")

In [ ]:
# Plot depth distribution across all CSVs
all_depths = []
for raw_file in raw_files:
    csv_path = OUTPUT_DIR / f"{raw_file.stem}_bottom.csv"
    if csv_path.exists():
        df_csv = pd.read_csv(csv_path)
        depths = df_csv['bottom_range_m'].dropna().values
        all_depths.append(depths)

if all_depths:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Histogram of all depths
    combined = np.concatenate(all_depths)
    axes[0].hist(combined, bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
    axes[0].set_xlabel('Bottom range from transducer (m)')
    axes[0].set_ylabel('Ping count')
    axes[0].set_title(f'Depth distribution (n={len(combined):,} pings)')
    axes[0].grid(True, alpha=0.3)

    # Per-file boxplot
    axes[1].boxplot(all_depths, labels=[f.stem[-6:] for f in raw_files[:len(all_depths)]],
                    patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6))
    axes[1].set_xlabel('File')
    axes[1].set_ylabel('Bottom range (m)')
    axes[1].set_title('Depth by file')
    axes[1].tick_params(axis='x', rotation=30)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## Notes

- Second-echo correction is active by default — if Sv at half-depth exceeds 75% of peak, the detector picks half depth instead to avoid locking onto the second echo.
- Sound velocity is hard-coded to 1500 m/s (actual ~1480–1490 m/s in cold fjord water), giving ~1% depth bias.
- `minimum_range=10.0 m` prevents transducer ring-down from being detected as bottom.
- Pings where no candidate exceeds the threshold return NaN — these should be excluded from training.
- The `'below'` mask is used as the U-Net segmentation target. A `line_thickness=3` for the line mask gives the optimizer more gradient signal near the boundary.